In [1]:
import sys
sys.path.append("../")

from models_training.segmentation.train_segmentation import (
    create_spark,
    load_data,
    cast_numeric_columns,
    assemble_features,
    scale_features,
    train_kmeans,
    evaluate_model,
    save_segmented_data,
    save_model
)

In [2]:
# creer spark
spark = create_spark()

26/02/22 22:07:39 WARN Utils: Your hostname, nouhayla-HP-EliteBook-835-G7-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.1.182 instead (on interface wlp1s0)
26/02/22 22:07:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/22 22:07:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/22 22:07:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/22 22:07:41 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
# load data
df = load_data(spark, "../data/silver/silver_data")
df.show(5)

+----------+---+------+------+---------------+------------+------------------+-------------------+--------------------+-------------+------------------+------------------+------------+----------+-----------+-----------------+-------------+-------------------+---------------+----------+
|CustomerID|Age|Gender|Income|CampaignChannel|CampaignType|           AdSpend|   ClickThroughRate|      ConversionRate|WebsiteVisits|     PagesPerVisit|        TimeOnSite|SocialShares|EmailOpens|EmailClicks|PreviousPurchases|LoyaltyPoints|AdvertisingPlatform|AdvertisingTool|Conversion|
+----------+---+------+------+---------------+------------+------------------+-------------------+--------------------+-------------+------------------+------------------+------------+----------+-----------+-----------------+-------------+-------------------+---------------+----------+
|      8074| 35|Female|144866|            PPC|   Awareness| 6808.222286805579|0.05946511349672668| 0.08844946865621445|           49|3.8115

In [4]:
df = cast_numeric_columns(df)
df.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Income: double (nullable = true)
 |-- CampaignChannel: string (nullable = true)
 |-- CampaignType: string (nullable = true)
 |-- AdSpend: string (nullable = true)
 |-- ClickThroughRate: string (nullable = true)
 |-- ConversionRate: string (nullable = true)
 |-- WebsiteVisits: string (nullable = true)
 |-- PagesPerVisit: string (nullable = true)
 |-- TimeOnSite: string (nullable = true)
 |-- SocialShares: string (nullable = true)
 |-- EmailOpens: string (nullable = true)
 |-- EmailClicks: string (nullable = true)
 |-- PreviousPurchases: double (nullable = true)
 |-- LoyaltyPoints: double (nullable = true)
 |-- AdvertisingPlatform: string (nullable = true)
 |-- AdvertisingTool: string (nullable = true)
 |-- Conversion: string (nullable = true)



In [5]:
df = assemble_features(df)
df.select("features").show(5, truncate=False)

+--------------------------+
|features                  |
+--------------------------+
|[35.0,144866.0,3483.0,8.0]|
|[25.0,67659.0,4643.0,0.0] |
|[34.0,135845.0,1558.0,1.0]|
|[46.0,28039.0,1960.0,7.0] |
|[50.0,80197.0,1356.0,7.0] |
+--------------------------+
only showing top 5 rows



In [6]:
# normalisation
df = scale_features(df)
df.select("scaled_features").show(5, truncate=False)


+---------------------------------------------------------------------------------+
|scaled_features                                                                  |
+---------------------------------------------------------------------------------+
|[-0.5787844214198687,1.6019473598183902,0.6944474553892024,1.2168928581485847]   |
|[-1.2497999236167088,-0.4525018947311132,1.5059045797490866,-1.5531008437119018] |
|[-0.6458859716395527,1.3619019400340304,-0.6521516518459505,-1.2068516309793411] |
|[0.15933263099665523,-1.5067752049912104,-0.37093978633502506,0.8706436454160239]|
|[0.42773883187539125,-0.11887042668363465,-0.7934571166051717,0.8706436454160239]|
+---------------------------------------------------------------------------------+
only showing top 5 rows



In [7]:
# train model
model = train_kmeans(df, k=4)

26/02/22 22:07:49 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


In [8]:
# metrics
metric = evaluate_model(model, df)
print("Silhouette score:", metric)

Silhouette score: 0.3038758910417426


In [9]:
# sauvgarde du new data 
save_segmented_data(model, df, "../data/silver/segment_data")

In [10]:
# save model
save_model(model, "../models_training/segmentation/segmentation_model")
print("segment model trained successfully")

segment model trained successfully


26/02/22 22:07:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
